# U.S. Presidential Inaugural Speech Analysis

**Recruiter-facing end-to-end analysis · Small-corpus exploratory NLP · Python 3.12/3.13**

> Kennedy 1961 and Nixon 1973 are the most similar tested pair by unigram/bigram TF-IDF cosine similarity (0.114), while topic and readability outputs remain exploratory.

## Executive summary

**Objective:** Compare lexical structure, readability, distinctive terms, similarity, and exploratory topics across three inaugural speeches.

**Data:** The 1941 Roosevelt, 1961 Kennedy, and 1973 Nixon inaugural addresses stored as three workbooks.

**Verified result:** Kennedy 1961 and Nixon 1973 are the most similar tested pair by unigram/bigram TF-IDF cosine similarity (0.114), while topic and readability outputs remain exploratory.

**Decision supported:** Locate lexical differences for closer human reading rather than automate historical judgment.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A digital-humanities or communications analyst.

**Decision:** Locate lexical differences for closer human reading rather than automate historical judgment.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '08-presidential-speech-analysis'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.10
pandas            2.3.3
NumPy             2.5.2
SciPy            1.18.0
scikit-learn      1.9.0
Matplotlib       3.11.1

Project: 08-presidential-speech-analysis


## 4. Data provenance and scope

The original project identifies NLTK's inaugural corpus as the source; local workbooks preserve imported snapshots.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

               file  size_mb           sha256
  kennedy_1961.xlsx    0.013 9798609bf906f09f
    nixon_1973.xlsx    0.013 2923240378678b5f
roosevelt_1941.xlsx    0.013 3e9a963d39efd24d


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


kennedy_1961.xlsx: 2 columns
   Name                                                                                               Speech
Kennedy Vice President Johnson, Mr. Speaker, Mr. Chief Justice, President Eisenhower, Vice President Nixon, 

nixon_1973.xlsx: 2 columns
 Name                                                                                               Speech
Nixon Mr. Vice President, Mr. Speaker, Mr. Chief Justice, Senator Cook, Mrs. Eisenhower, and my fellow cit

roosevelt_1941.xlsx: 2 columns
     Name                                                                                               Speech
Roosevelt On each national day of inauguration since 1789, the people have renewed their sense of dedication t


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 113 lines
Functions: _syllables, _document_metrics, run_analysis


## 7. Methodology and hypotheses

Corpus-quality audit, tokenization, lexical diversity, approximate Flesch readability, unigram/bigram TF-IDF, cosine similarity, distinctive-term analysis, and sentence-level NMF topics.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_08_presidential_speech_analysis", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 0.49 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (6 fields)
      document column  dtype  missing_count  missing_percent  unique_values  constant
Roosevelt 1941   Name object              0              0.0              1      True
Roosevelt 1941 Speech object              0              0.0              1      True
  Kennedy 1961   Name object              0              0.0              1      True
  Kennedy 1961 Speech object              0              0.0              1      True
    Nixon 1973   Name object              0              0.0              1      True
    Nixon 1973 Speech object              0              0.0              1      True


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'distinctive_tfidf_terms.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'Kennedy 1961 and Nixon 1973 are the most similar tested pair by unigram/bigram TF-IDF cosine similarity (0.114), while topic and readability outputs remain exploratory.')

Primary evidence: distinctive_tfidf_terms.csv, shape=(45, 3)
      document          term  tfidf
Roosevelt 1941     democracy 0.0991
Roosevelt 1941          body 0.0809
Roosevelt 1941          mind 0.0809
Roosevelt 1941        speaks 0.0809
Roosevelt 1941          know 0.0778
Roosevelt 1941        spirit 0.0753
Roosevelt 1941         years 0.0658
Roosevelt 1941   nation like 0.0650
Roosevelt 1941  spirit faith 0.0650
Roosevelt 1941          came 0.0650
Roosevelt 1941   like person 0.0650
Roosevelt 1941          like 0.0650
Roosevelt 1941       destiny 0.0650
Roosevelt 1941 united states 0.0650
Roosevelt 1941     continent 0.0650

Verified result:
Kennedy 1961 and Nixon 1973 are the most similar tested pair by unigram/bigram TF-IDF cosine similarity (0.114), while topic and readability outputs remain exploratory.


## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])

## 12. Visual evidence

### Speech Analysis Evidence

![speech_analysis_evidence](../reports/figures/speech_analysis_evidence.png)

### Top Words By Speech

![top_words_by_speech](../reports/figures/top_words_by_speech.png)

## 13. Business interpretation

Kennedy 1961 and Nixon 1973 are the most similar tested pair by unigram/bigram TF-IDF cosine similarity (0.114), while topic and readability outputs remain exploratory.

The correct action is to use this result as evidence for **Locate lexical differences for closer human reading rather than automate historical judgment.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

Lexical counts, similarity, readability, and topics do not establish ideology, truthfulness, policy effect, or speaker intent.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                    artifact  size_kb       sha256
reports\figures\speech_analysis_evidence.png    260.2 2bcfbcbaa5d0
     reports\figures\top_words_by_speech.png     82.0 23a05c86f056
                        reports\metrics.json      7.4 e2c7afc197b7
           reports\tables\corpus_metrics.csv      0.4 45f76911fdb2
             reports\tables\data_quality.csv      0.3 b8b61b4769e9
  reports\tables\distinctive_tfidf_terms.csv      1.9 793fc707fb3a
       reports\tables\exploratory_topics.csv      0.3 786d1a99da65
         reports\tables\tfidf_similarity.csv      0.3 c7a257a29da3
    reports\tables\topic_share_by_speech.csv      0.2 940c907a1e95


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed compare lexical structure, readability, distinctive terms, similarity, and exploratory topics across three inaugural speeches. using corpus-quality audit, tokenization, lexical diversity, approximate flesch readability, unigram/bigram tf-idf, cosine similarity, distinctive-term analysis, and sentence-level nmf topics. The final verified conclusion is: **Kennedy 1961 and Nixon 1973 are the most similar tested pair by unigram/bigram TF-IDF cosine similarity (0.114), while topic and readability outputs remain exploratory.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/08-presidential-speech-analysis/src/analysis.py
python scripts/execute_notebooks.py --project 08-presidential-speech-analysis
```